In [16]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoConfig, AutoModel
from bert_utils_local import chemberta2_embed

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
MODEL_NAME = "DeepChem/ChemBERTa-77M-MLM"
device='cpu'

bert_embed = chemberta2_embed(name=MODEL_NAME, device=device)
TOKENIZER = bert_embed.tokenizer
config = bert_embed.config
CHEMBERT_MODEL = bert_embed.model

In [18]:
# tokens = bert_embed.smiles_list2tokens(df['smiles'])
# tokens = {k: v.to(device) for k, v in tokens.items()}
# out = CHEMBERT_MODEL(**tokens)
# out.last_hidden_state.shape
# tokens['attention_mask'].unsqueeze(-1).shape

In [19]:
smiles = "BrNc1nc2c(ncn2[C@H]2C=C[C@@H](CO)C2)c(NC2CC2)n1"

tokens_out = bert_embed.smiles2tokens(smiles)
tokens = tokens_out['tokens']

reconstructed = bert_embed.tokens2smiles(tokens_out['encoded']["input_ids"])
print("Original:     ", smiles)
print("Tokens:       ", tokens)
print("Reconstructed:", reconstructed)

Original:      BrNc1nc2c(ncn2[C@H]2C=C[C@@H](CO)C2)c(NC2CC2)n1
Tokens:        ['[CLS]', 'B', 'N', 'c', '1', 'n', 'c', '2', 'c', '(', 'n', 'c', 'n', '2', 'C', '2', 'C', '=', 'C', 'C', '(', 'C', 'O', ')', 'C', '2', ')', 'c', '(', 'N', 'C', '2', 'C', 'C', '2', ')', 'n', '1', '[SEP]']
Reconstructed: BNc1nc2c(ncn2C2C=CC(CO)C2)c(NC2CC2)n1


In [20]:
# z = bert_embed.embed_smiles(smiles_list=df['smiles'])
# z.shape

## **Test the model for representation**

In [21]:
import selfies as sf
from rdkit import Chem

smiles2selfie_fn = lambda x: sf.encoder(x)
canonical_fn = lambda smi: Chem.MolToSmiles(Chem.MolFromSmiles(smi), isomericSmiles=False)
smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)
len_filter_fn = lambda x: sf.len_selfies(x)<=128

#### **Preprocess drugs**

In [22]:
from bert_utils_local import check_smiles_validity

df = pd.read_csv('/home/rkmvu/Dataset/selfies/drug/mdrug.csv')
_, valid_mask = check_smiles_validity(smiles=df['smiles'].tolist())
df = df[valid_mask].copy()
df['smiles'] = df['smiles'].apply(canonical_fn)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

len_mask = df['selfies'].apply(len_filter_fn)
df = df[len_mask]

df_temp = df.copy()
mask = df_temp['smiles'].apply(smiles_clean_fn)
df_temp = df_temp[mask]
df_temp = df_temp.drop_duplicates(subset=['smiles'])
smiles_main = df_temp['smiles'].tolist()
print(f'Number of smiles: {len(smiles_main)}')
df_temp

<===checking smiles validity===>


100%|██████████| 1381/1381 [00:00<00:00, 4788.64it/s]

Number of smiles: 1089


,name,smiles,InChl,type,selfies
0,Abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,InChI=1S/C14H18N6O/c15-14-18-12(17-9-2-3-9)11-...,Drug,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...
1,Abiraterone,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,InChI=1S/C26H33NO2/c1-17(28)29-20-10-12-25(2)1...,Drug,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...
2,Acamprosate,CC(=O)NCCCS(=O)(=O)O,"InChI=1S/C5H11NO4S/c1-5(7)6-3-2-4-11(8,9)10/h2...",Drug,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...
3,Acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,InChI=1S/C25H43NO18/c1-6-11(26-8-2-7(3-27)12(3...,Drug,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...
4,Acebutolol,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,InChI=1S/C18H28N2O4/c1-5-6-18(23)20-14-7-8-17(...,Drug,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...
...,...,...,...,...,...
1374,Ziprasidone,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,InChI=1S/C21H21ClN4OS/c22-17-13-18-15(12-20(27...,Drug,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...
1375,Zoledronate,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,"InChI=1S/C5H10N2O7P2/c8-5(15(9,10)11,16(12,13)...",Drug,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...
1377,Zolpidem,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,InChI=1S/C19H21N3O/c1-13-5-8-15(9-6-13)19-16(1...,Drug,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...
1378,Zonisamide,NS(=O)(=O)Cc1noc2ccccc12,"InChI=1S/C8H8N2O3S/c9-14(11,12)5-7-6-3-1-2-4-8...",Drug,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...


### **Finding Nearest Neighbours**

In [23]:

def encode(smiles_or_selfies):
    z =   bert_embed.embed_smiles(smiles_list=smiles_or_selfies)
    return z

def name2smiles(name):
    row = df_temp[df_temp['name'] == name].iloc[0]
    return row['smiles']

def smiles2name(smiles):
    row = df_temp[df_temp['smiles'] == smiles].iloc[0]
    return row['name']

def smi2nneigh(smiles, indices, n_neigh=10):
    idx = smiles_main.index(smiles)
    neigh_idx = indices[idx][1:n_neigh+1]
    nn_smiles = [smiles_main[i] for i in neigh_idx]
    nn_names = [smiles2name(x) for x in nn_smiles]
    return {'smiles':nn_smiles, 'name':nn_names}


In [24]:
from sklearn.neighbors import NearestNeighbors

z = encode(smiles_or_selfies=smiles_main)
n_neigh = NearestNeighbors(n_neighbors=60, metric='euclidean')
n_neigh.fit(z)

NearestNeighbors(metric='euclidean', n_neighbors=60)

In [25]:
distances, indices = n_neigh.kneighbors(z)
distances
indices

array([[   0,  481,  387, ...,  708,  463, 1040],
       [   1,  674,  521, ...,   39,  320, 1049],
       [   2,   12,  166, ...,  500,  925,  975],
       ...,
       [1086, 1081,  681, ...,  764,  818,  596],
       [1087,  397,  965, ...,  554,  242,  752],
       [1088,  787,  869, ...,  138,  879,  481]])

In [26]:
name2smiles('Abacavir')

'Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1'

In [27]:
smiles2name(name2smiles('Abacavir'))

'Abacavir'

In [28]:
smi2nneigh(name2smiles('Loxapine'), indices=indices)

{'smiles': ['CN1CCN(C2=Nc3cc(Cl)ccc3Nc3ccccc32)CC1',
  'CN1CCN(CCCN2c3ccccc3Sc3ccc(Cl)cc32)CC1',
  'CCN(CC)CCNc1ccc(C)c2sc3ccccc3c(=O)c12',
  'OCCOCCN1CCN(C2=Nc3ccccc3Sc3ccccc32)CC1',
  'OCCN1CCN(CCCN2c3ccccc3Sc3ccc(Cl)cc32)CC1',
  'CN1CCN(CC(=O)N2c3ccccc3C(=O)Nc3cccnc32)CC1',
  'CN1CCCC(n2nc(Cc3ccc(Cl)cc3)c3ccccc3c2=O)CC1',
  'CN1CCC(=C2c3ccccc3CC(=O)c3sccc32)CC1',
  'OCCN1CCN(CCC=C2c3ccccc3Sc3ccc(Cl)cc32)CC1',
  'CN1CCC(=C2c3ccccc3C=Cc3ccccc32)CC1'],
 'name': ['Clozapine',
  'Prochlorperazine',
  'Lucanthone',
  'Quetiapine',
  'Perphenazine',
  'Pirenzepine',
  'Azelastine',
  'Ketotifen',
  'Zuclopenthixol',
  'Cyproheptadine']}

In [29]:
df = pd.DataFrame(z, columns=[f'dim_{i}' for i in range(z.shape[1])])
df.insert(0, 'smiles', smiles_main)
df.insert(0, 'selfies', 'None')
df.insert(0, 'name', 'None')
df['name'] = df['smiles'].apply(smiles2name)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

df2 = pd.read_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_properties.csv')
df3 = pd.merge(df, df2, on='smiles')
# df3.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_props_with_embeds_bert.csv', index=False)
df3#.head(2)

,name,selfies,smiles,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,...,VSA_EState3,NHOHCount,NumHDonors,NumHAcceptor,NumRotatableBonds,MolLogP,ATSC1pe,ATSC1are,AATSC1dv,AATSC1are
0,Abacavir,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,-0.458140,0.199744,-1.356518,-0.101498,0.099398,-0.483272,0.830083,...,12.615719,4,3,7,4,1.09230,-0.478300,-0.657070,1.039823,-0.015645
1,Abiraterone,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,-0.467864,0.415050,-1.355940,-0.234235,0.072859,-0.527838,0.975104,...,0.000000,0,0,3,2,5.96940,0.296168,0.241524,1.157286,0.003659
2,Acamprosate,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...,CC(=O)NCCCS(=O)(=O)O,-0.066814,0.355068,-1.529787,-0.014106,0.213245,-0.714304,0.990866,...,2.403611,2,2,3,4,-0.59960,-0.423500,-0.761825,-0.369321,-0.036277
3,Acarbose,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,-0.302002,0.389767,-1.355805,0.061697,0.226347,-0.482877,1.081419,...,135.592960,14,14,19,9,-8.56450,-4.505442,-5.305658,-0.192650,-0.058952
4,Acebutolol,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,-0.351344,0.462917,-1.352025,-0.057216,0.214585,-0.476913,0.942681,...,15.766019,3,3,5,10,2.36550,-0.276185,-0.373677,1.298077,-0.007186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1084,Ziprasidone,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,-0.609919,0.241661,-1.227277,-0.061279,0.249783,-0.353973,0.826396,...,4.859700,1,1,5,4,3.80900,0.140586,0.027960,0.820617,0.000528
1085,Zoledronate,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,-0.195782,0.204714,-1.385482,0.336822,0.379866,-0.540558,0.984859,...,6.037515,5,5,5,4,-1.11540,-3.912300,-4.806854,-2.436445,-0.184879
1086,Zolpidem,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,-0.607467,0.252180,-1.221335,-0.038693,0.348454,-0.340394,0.802913,...,0.000000,0,0,3,3,3.24884,0.303576,0.245848,1.704815,0.005345
1087,Zonisamide,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...,NS(=O)(=O)Cc1noc2ccccc12,-0.645616,0.097150,-1.364970,0.048231,0.325350,-0.458763,0.765496,...,9.236725,2,1,4,2,0.61630,0.064800,-0.113569,0.317683,-0.004938


## **Nearest Neighbour Analysis**

In [30]:
import numpy as np

top_drugs = ["Metformin", "Amoxicillin", "Atorvastatin", "Amlodipine", "Acetaminophen", "Imatinib", "Clozapine", 
             "Ibuprofen", "Azithromycin", "Doxycycline"]
top_drugs = df_temp['name']
n_neigh=50

all_out = {'drug_bert':[], 'nn_idx_bert':[], 'smiles_bert':[], 'names_bert':[]}
for drug in top_drugs:
    _smiles = name2smiles(drug)
    out = smi2nneigh(_smiles, indices=indices, n_neigh=n_neigh)
    drug_name = [drug]*n_neigh
    idx = np.arange(1, n_neigh+1)
    out = {'drug':drug_name, 'nn_idx':idx, **out}
    all_out['drug_bert'].extend(out['drug'])
    all_out['nn_idx_bert'].extend(out['nn_idx'])
    all_out['smiles_bert'].extend(out['smiles'])
    all_out['names_bert'].extend(out['name'])

In [ ]:
all_out_df = pd.DataFrame(all_out)
# all_out_df.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/near_smiles_bert.csv', index=False)
all_out_df

,drug_bert,nn_idx_bert,smiles_bert,names_bert
0,Abacavir,1,CN1C2CCCC1CC(NC(=O)c1nn(C)c3ccccc13)C2,Granisetron
1,Abacavir,2,CN1CCN(C(=O)OC2c3nccnc3C(=O)N2c2ccc(Cl)cn2)CC1,Eszopiclone
2,Abacavir,3,CC(=O)OCC(CCn1cnc2cnc(N)nc21)COC(C)=O,Famciclovir
3,Abacavir,4,CC(=O)CCCCn1c(=O)c2c(ncn2C)n(C)c1=O,Pentoxifylline
4,Abacavir,5,COc1cc2nc(N(C)CCCNC(=O)C3CCCO3)nc(N)c2cc1OC,Alfuzosin
...,...,...,...,...
54445,Zuclopenthixol,46,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,Erlotinib
54446,Zuclopenthixol,47,CN1CCN(CCC=C2c3ccccc3Sc3ccc(S(=O)(=O)N(C)C)cc3...,Thiothixene
54447,Zuclopenthixol,48,COC12CC(COC(=O)c3cncc(Br)c3)CN(C)C1Cc1cn(C)c3c...,Nicergoline
54448,Zuclopenthixol,49,CN(C)CCCC1(c2ccc(F)cc2)OCc2cc(C#N)ccc21,Citalopram
